# 365 Probabilidades - Dia #113
## Qual a probabilidade de um abraço te proteger de um resfriado?

**Tipo:** Emocional
**Data de publicação:** 2026-10-04
**Ferramenta:** Python
**Decisão analisada:** Vale a pena insistir no contato físico quando a vida aperta?
**Hashtag:** #365Probabilidades #Dia113

---

### 📖 A História

Existe um estudo em que pesquisadores expuseram 404 pessoas a vírus respiratórios de
propósito.

Gotas nasais com rinovírus ou influenza, e depois cinco ou seis dias de quarentena, com
a infecção medida em laboratório.

Antes disso, durante catorze noites seguidas, cada uma dessas pessoas atendeu o telefone
e respondeu duas coisas: se tinha tido algum atrito naquele dia, e se tinha recebido
abraço.

A pergunta não era se abraço faz bem. Era outra, mais específica: quando a vida aperta,
ser cuidado amortece o efeito no corpo?

E este é o primeiro dia do projeto em que **os dados originais foram baixados e o modelo
publicado foi refeito do zero**, em vez de citado.

---

### 📚 O Conceito: amortecimento, não proteção

A hipótese do estudo tem nome: *stress buffering*, amortecimento de estresse.

Ela não diz que apoio social protege sempre. Diz que apoio social **muda a inclinação**
da relação entre adversidade e adoecimento. Quem tem apoio sofre menos com o mesmo
atrito; quem não tem, sofre mais.

Isso é uma afirmação sobre uma interação, não sobre um efeito principal. E é por isso que
o número que a manchete costuma carregar, o de "quem abraça mais adoece menos", não é o
resultado central do artigo.

O resultado central é o que acontece com a inclinação.

---

### 🧮 O Modelo

Quatro camadas, todas feitas sobre os dados individuais depositados pelos autores.

1. **O que aconteceu com as 404 pessoas.** Exposição, infecção e doença clínica são três
   coisas diferentes, e o estudo separa as três.
2. **Reprodução dos modelos publicados.** Regressão logística ajustada por catorze
   covariáveis, exatamente como na sintaxe depositada, conferida coeficiente a
   coeficiente contra o artigo.
3. **A Figura 4 do artigo, refeita.** Probabilidade prevista de infecção conforme
   aumentam os dias de atrito, para quem foi pouco e para quem foi muito abraçado.
   É aqui que o amortecimento aparece.
4. **A atenuação.** Quanto o coeficiente do apoio percebido encolhe quando a frequência
   de abraços entra no mesmo modelo.

**Fonte:**
- Sheldon Cohen, Janicki-Deverts, D., Turner, R. B. & Doyle, W. J. (2015). "Does Hugging
  Provide Stress-Buffering Social Support? A Study of Susceptibility to Upper
  Respiratory Infection and Illness". *Psychological Science* 26(2), 135-147.
- **Dados individuais:** Dryad, doi:10.5061/dryad.g7b40. Arquivo de replicação em SPSS
  com 57 variáveis, mais a sintaxe que reproduz as análises do manuscrito e o dicionário
  de variáveis. Convertido aqui para `dados113/hugging2015_analitico.csv`, com as 404
  linhas da amostra analítica e as 22 variáveis usadas.

**Regra do nome completo:** escrever sempre **Sheldon Cohen**. Randy Cohen, cardiologista,
assina fontes do **#064** e do **#092**, e são pessoas diferentes.

**Nota metodológica:** o ×0,80 não se aplica. A infecção foi medida em laboratório depois
de exposição controlada. Abraço e atrito são autorrelatados, em catorze ligações, mas são
a exposição, não o desfecho. Autorrelato de comportamento diário específico não é a mesma
coisa que autorrelato de percepção geral, e nenhum dos dois é medida objetiva.

**Linguagem:** tudo aqui é associação. O desenho aleatorizou o vírus, não os abraços.

**Ponte interna:** o **#070** tratou da fenda entre o que a pessoa relata e o que o
instrumento mede. Aqui as duas medidas estão dentro do mesmo estudo e nas mesmas pessoas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · só numpy, sem pacote de modelagem")

In [ ]:
# --- OS DADOS INDIVIDUAIS ---
# Cohen, Janicki-Deverts, Turner & Doyle (2015). Dryad doi:10.5061/dryad.g7b40.
# Amostra analítica: as 404 linhas com massa corporal presente, como no filtro da
# sintaxe original dos autores.

ARQUIVO = 'dados113/hugging2015_analitico.csv'

with open(ARQUIVO, encoding='utf-8') as f:
    linhas = list(csv.DictReader(f))

def coluna(nome):
    return np.array([float(l[nome]) if l[nome] not in ('', 'nan') else np.nan
                     for l in linhas])

infectado = coluna('post_infected')
resfriado = coluna('post_objcold')
pcthug = coluna('pcthug')          # fração dos 14 dias com abraço
pctten = coluna('pctten')          # fração dos 14 dias com atrito
cpcthug = coluna('cpcthug')        # as duas acima, centradas
cpctten = coluna('cpctten')
cisel12tot = coluna('cisel12tot')  # apoio percebido, centrado

COVARIAVEIS = ['age', 'sex', 'study_id', 'race_white', 'pre_ab_ge4', 'bodymass',
               'spring', 'summer', 'fall', 'virus', 'educ_hschl', 'educ_lt2yr',
               'educ_assoc', 'sni_hcr_married']
COV = np.column_stack([coluna(c) for c in COVARIAVEIS])

N = len(linhas)
DP_HUG = cpcthug.std(ddof=1)

print("=" * 70)
print("  OS DADOS ORIGINAIS, BAIXADOS E LIDOS")
print("=" * 70)
print(f"\n  Amostra analítica:            {N} pessoas")
print(f"  Expostas ao vírus:            {N} (todas)")
print(f"  Desenvolveram infecção:       {int(infectado.sum())}  "
      f"({infectado.mean() * 100:.1f}%)")
print(f"  Desenvolveram resfriado clínico: {int(np.nansum(resfriado))}  "
      f"({np.nanmean(resfriado) * 100:.1f}%)")
print(f"\n  Exposição, infecção e doença são três coisas diferentes.")
print(f"\n  Dias com abraço, média:       {pcthug.mean() * 100:.1f}% dos 14 dias "
      f"(desvio padrão {DP_HUG * 100:.1f} pontos)")
print(f"  Dias com atrito, média:       {pctten.mean() * 100:.1f}% dos 14 dias")
print(f"  Metade das pessoas teve atrito em até {np.percentile(pctten, 50) * 100:.1f}% "
      f"dos dias; 10% passaram de {np.percentile(pctten, 90) * 100:.1f}%.")
print(f"\n  ×0,80: não se aplica. Infecção medida em laboratório.")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# Regressão logística por mínimos quadrados reponderados. Escrita aqui para o
# notebook não depender de pacote de modelagem, e conferida contra o artigo.
def logistica(X, y):
    X = np.asarray(X, float)
    b = np.zeros(X.shape[1])
    for _ in range(200):
        eta = X @ b
        mu = 1 / (1 + np.exp(-eta))
        w = np.clip(mu * (1 - mu), 1e-10, None)
        z = eta + (y - mu) / w
        XtW = X.T * w
        novo = np.linalg.solve(XtW @ X, XtW @ z)
        if np.max(np.abs(novo - b)) < 1e-11:
            b = novo
            break
        b = novo
    eta = X @ b
    mu = 1 / (1 + np.exp(-eta))
    cov = np.linalg.inv((X.T * np.clip(mu * (1 - mu), 1e-10, None)) @ X)
    return b, cov

def monta(*extras):
    return np.column_stack([np.ones(N), COV] + list(extras))

hugXten = cpcthug * cpctten
iselXten = cisel12tot * cpctten

# ── CAMADA 2 · reprodução dos modelos publicados ──
b2, cov2 = logistica(monta(cpcthug, cpctten, hugXten), infectado)
ep2 = np.sqrt(np.diag(cov2))
i_hug, i_ten, i_int = -3, -2, -1

b_sem, cov_sem = logistica(monta(cpcthug, cpctten), infectado)   # sem a interação
ep_sem = np.sqrt(np.diag(cov_sem))

b1, cov1 = logistica(monta(cisel12tot, cpctten, iselXten), infectado)
ep1 = np.sqrt(np.diag(cov1))

def ic(b, ep, i):
    return b[i] - 1.96 * ep[i], b[i] + 1.96 * ep[i]

print("=" * 70)
print("  CAMADA 2 · O MODELO PUBLICADO, REFEITO DO ZERO")
print("=" * 70)
print("\n  Todos ajustados pelas mesmas 14 covariáveis da sintaxe original.\n")
lo, hi = ic(b2, ep2, i_int)
print(f"  Interação atrito × abraço")
print(f"    aqui:      b = {vir(b2[i_int], 2)}   IC 95% [{vir(lo, 2)}; {vir(hi, 2)}]")
print(f"    no artigo: b = -12,31        IC 95% [-20,94; -3,69]")
lo, hi = ic(b_sem, ep_sem, -2)
print(f"\n  Abraço, efeito principal, no modelo SEM interação")
print(f"    aqui:      razão de chances {vir(np.exp(b_sem[-2]), 2)}   "
      f"IC 95% [{vir(np.exp(lo), 2)}; {vir(np.exp(hi), 2)}]")
print(f"    no artigo: razão de chances 0,39        IC 95% [0,16; 0,96]")
lo, hi = ic(b1, ep1, i_int)
print(f"\n  Interação atrito × apoio percebido")
print(f"    aqui:      b = {vir(b1[i_int], 2)}    IC 95% [{vir(lo, 2)}; {vir(hi, 2)}]")
print(f"    no artigo: b = -0,40         IC 95% [-0,79; -0,003]")
print(f"\n  Os coeficientes batem. O que vem a seguir é conta nossa sobre o modelo deles.")
print(f"\n  ATENÇÃO à razão de chances de 0,39: ela vem do modelo SEM a interação.")
print(f"  No modelo completo, com a interação, o mesmo efeito principal é "
      f"{vir(np.exp(b2[i_hug]), 2)}.")
print(f"  Não são números concorrentes: significam coisas diferentes.")

# ── CAMADA 3 · a Figura 4 do artigo, refeita ──
media_cov = COV.mean(axis=0)
media_ten = pctten.mean()

def prevista(ten, hug_centrado):
    tc = ten - media_ten
    v = np.r_[1.0, media_cov, hug_centrado, tc, hug_centrado * tc]
    lin = v @ b2
    se = np.sqrt(v @ cov2 @ v)
    f = lambda x: 1 / (1 + np.exp(-x))
    return f(lin), f(lin - 1.96 * se), f(lin + 1.96 * se)

LIM_TEN = np.percentile(pctten, 90)   # até onde há gente suficiente

print("\n" + "=" * 70)
print("  CAMADA 3 · A FIGURA 4 DO ARTIGO, REFEITA")
print("=" * 70)
print(f"\n  Probabilidade prevista de infecção, covariáveis na média.")
print(f"  Pouco abraçado = 1 desvio abaixo da média; muito abraçado = 1 desvio acima.\n")
print(f"  {'dias com atrito':>16}{'pouco abraçado':>20}{'muito abraçado':>20}")
for ten in [0.0, 0.07, 0.14, 0.21, LIM_TEN]:
    p_lo = prevista(ten, -DP_HUG)
    p_hi = prevista(ten, +DP_HUG)
    print(f"  {ten * 100:14.0f}%{vir(p_lo[0] * 100) + '%':>20}{vir(p_hi[0] * 100) + '%':>20}")
z_lo0, z_hi0 = prevista(0.0, -DP_HUG)[0], prevista(0.0, DP_HUG)[0]
z_loF, z_hiF = prevista(LIM_TEN, -DP_HUG)[0], prevista(LIM_TEN, DP_HUG)[0]
print(f"\n  Sem atrito nenhum, abraço não separa nada: "
      f"{vir(z_lo0 * 100)}% contra {vir(z_hi0 * 100)}%.")
print(f"  Com atrito em {vir(LIM_TEN * 100, 0)}% dos dias: "
      f"{vir(z_loF * 100)}% contra {vir(z_hiF * 100)}%, "
      f"distância de {vir(abs(z_loF - z_hiF) * 100)} pontos.")
print(f"\n  É isso que amortecimento quer dizer: não é um efeito, é uma inclinação.")

# ── CAMADA 4 · a atenuação ──
b4, cov4 = logistica(monta(cpcthug, cpctten, cisel12tot, iselXten, hugXten), infectado)
ep4 = np.sqrt(np.diag(cov4))
j_isel, j_hug = -2, -1
atenuacao = (1 - b4[j_isel] / b1[i_int]) * 100

print("\n" + "=" * 70)
print("  CAMADA 4 · QUANTO DO APOIO PERCEBIDO É ABRAÇO")
print("=" * 70)
lo, hi = ic(b1, ep1, i_int)
print(f"\n  Interação do apoio percebido, sozinha:")
print(f"    b = {vir(b1[i_int], 3)}   IC 95% [{vir(lo, 3)}; {vir(hi, 3)}]")
lo, hi = ic(b4, ep4, j_isel)
print(f"  A mesma, com o abraço no modelo:")
print(f"    b = {vir(b4[j_isel], 3)}   IC 95% [{vir(lo, 3)}; {vir(hi, 3)}]")
lo, hi = ic(b4, ep4, j_hug)
print(f"  E a do abraço, no mesmo modelo:")
print(f"    b = {vir(b4[j_hug], 2)}  IC 95% [{vir(lo, 2)}; {vir(hi, 2)}]")
print(f"\n  Atenuação do coeficiente do apoio: {vir(atenuacao)}%")
print(f"  O artigo reporta 32%.")
print(f"\n  DIVERGÊNCIA REGISTRADA. Todos os coeficientes acima reproduzem o artigo")
print(f"  com três casas. A fração de atenuação que eles implicam é {vir(atenuacao)}%,")
print(f"  não 32%. Testamos as variantes plausíveis do modelo e nenhuma chega a 32%.")
print(f"  Não sabemos como os autores calcularam. Reportamos os dois números.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · exposição, infecção, doença ══
fig1, ax1 = plt.subplots(figsize=(11, 6.5))
etapas = ['expostas ao vírus', 'desenvolveram infecção', 'desenvolveram resfriado clínico']
valores = [N, int(infectado.sum()), int(np.nansum(resfriado))]
b = ax1.barh(range(3), valores, color=[CINZA, DOURADO, VERMELHO], alpha=0.88, height=0.55)
for i, v in enumerate(valores):
    ax1.text(v + 7, i, f'{v}   ({v / N * 100:.1f}%)', va='center', fontsize=12.5,
             fontweight='bold')
ax1.set_yticks(range(3))
ax1.set_yticklabels(etapas, fontsize=12)
ax1.invert_yaxis()
ax1.set_xlim(0, N * 1.3)
ax1.set_xlabel('Pessoas')
ax1.set_title('Três coisas diferentes, e o estudo separa as três\n'
              f'Sheldon Cohen et al. (2015) · amostra analítica de {N} pessoas',
              fontsize=13, pad=15)
ax1.grid(axis='y', alpha=0)
plt.figtext(0.5, 0.005, 'Dados individuais: Dryad doi:10.5061/dryad.g7b40 | '
            '#365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-113-grafico-01-o-estudo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a reprodução ══
fig2, ax2 = plt.subplots(figsize=(11, 6.5))
itens = [
    ('interação\natrito × abraço', b2[i_int], np.sqrt(np.diag(cov2))[i_int], -12.31, 5.0),
    ('interação\natrito × apoio percebido', b1[i_int], np.sqrt(np.diag(cov1))[i_int],
     -0.40, 0.22),
]
esc = [5.0, 0.22]
ys = [1, 0]
for (rot, bn, epn, bp, _), y, e in zip(itens, ys, esc):
    ax2.errorbar(bn / e, y + 0.13, xerr=1.96 * epn / e, fmt='o', color=VERDE,
                 markersize=11, capsize=6, elinewidth=2, label='refeito aqui' if y == 1 else '')
    ax2.plot(bp / e, y - 0.13, 's', color=VERMELHO, markersize=11,
             label='publicado' if y == 1 else '')
    ax2.text(bn / e, y + 0.33, f'b = {vir(bn, 2)}', ha='center', fontsize=10.5, color=VERDE)
    ax2.text(bp / e, y - 0.42, f'b = {vir(bp, 2)}', ha='center', fontsize=10.5,
             color=VERMELHO)
ax2.axvline(0, color='#333333', linewidth=1.5)
ax2.set_yticks(ys)
ax2.set_yticklabels([i[0] for i in itens], fontsize=11.5)
ax2.set_ylim(-0.75, 1.75)
ax2.set_xlabel('Coeficiente, cada linha na sua própria escala')
ax2.set_title('Os dois coeficientes centrais do artigo, refeitos dos dados originais\n'
              'Barras são IC 95%. A linha vertical é o zero',
              fontsize=13, pad=15)
ax2.legend(frameon=False, fontsize=11, loc='lower right')
ax2.set_xticks([])
ax2.grid(axis='y', alpha=0)
plt.figtext(0.5, 0.005, 'Regressão logística ajustada por 14 covariáveis, como na '
            'sintaxe depositada | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-113-grafico-02-a-reproducao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · o amortecimento (Figura 4 refeita) ══
fig3, ax3 = plt.subplots(figsize=(11, 6.5))
tens = np.linspace(0, LIM_TEN, 120)
for hug_c, cor, rot in [(-DP_HUG, VERMELHO, 'pouco abraçado'),
                        (+DP_HUG, VERDE, 'muito abraçado')]:
    p = np.array([prevista(t, hug_c) for t in tens])
    ax3.plot(tens * 100, p[:, 0] * 100, color=cor, linewidth=3, label=rot)
    ax3.fill_between(tens * 100, p[:, 1] * 100, p[:, 2] * 100, color=cor, alpha=0.15)
ax3.set_xlabel('Dias com atrito, em % dos 14 dias registrados')
ax3.set_ylabel('Probabilidade prevista de infecção (%)')
ax3.set_ylim(50, 100)
ax3.set_title('Sem atrito, o abraço não separa nada. Com atrito, separa muito.\n'
              'Figura 4 do artigo, refeita a partir dos dados depositados',
              fontsize=13, pad=15)
ax3.legend(frameon=False, fontsize=12, loc='upper left')
ax3.annotate('', xy=(LIM_TEN * 100, z_hiF * 100), xytext=(LIM_TEN * 100, z_loF * 100),
             arrowprops=dict(arrowstyle='<->', color=DOURADO, lw=2.2))
ax3.text(LIM_TEN * 100 - 1.5, (z_loF + z_hiF) / 2 * 100,
         f'{vir(abs(z_loF - z_hiF) * 100)} pontos', ha='right', fontsize=12,
         color=DOURADO, fontweight='bold')
plt.figtext(0.5, 0.005, 'Modelo dos autores, refeito · covariáveis na média · faixa '
            'até o percentil 90 de atrito | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-113-grafico-03-o-amortecimento.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a atenuação ══
fig4, ax4 = plt.subplots(figsize=(11, 6.5))
eps1 = np.sqrt(np.diag(cov1))[i_int]
eps4 = np.sqrt(np.diag(cov4))[j_isel]
vals = [(b1[i_int], eps1, 'apoio percebido\nsozinho', DOURADO),
        (b4[j_isel], eps4, 'apoio percebido\ncom o abraço no modelo', CINZA)]
for i, (bv, ev, rot, cor) in enumerate(vals):
    y = 1 - i
    ax4.errorbar(bv, y, xerr=1.96 * ev, fmt='o', color=cor, markersize=13,
                 capsize=7, elinewidth=2.4)
    ax4.text(bv, y + 0.22, f'b = {vir(bv, 3)}', ha='center', fontsize=12, color=cor)
ax4.axvline(0, color='#333333', linewidth=1.8)
ax4.set_yticks([1, 0])
ax4.set_yticklabels([v[2] for v in vals], fontsize=11.5)
ax4.set_ylim(-0.6, 1.7)
ax4.set_xlabel('Coeficiente da interação entre atrito e apoio percebido')
ax4.set_title('Parte do amortecimento do apoio percebido também aparece nos abraços\n'
              f'O coeficiente encolhe {vir(atenuacao)}% e o intervalo passa a cruzar o zero',
              fontsize=13, pad=15)
ax4.text(b4[j_isel] + 0.02, -0.42,
         f'aqui: {vir(atenuacao)}%   ·   no artigo: 32%',
         fontsize=11.5, color=VERMELHO, ha='center')
plt.figtext(0.5, 0.005, 'Modelo dos autores, refeito · a divergência de atenuação está '
            'declarada nas limitações | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-113-grafico-04-a-atenuacao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

Quatrocentas e quatro pessoas receberam vírus no nariz, e a pergunta não era se abraço
faz bem. Era se ele muda o que o atrito faz com o corpo.

Refazendo o modelo dos autores a partir dos dados que eles depositaram, a resposta
aparece limpa.

**Entre quem não teve atrito nenhum nas duas semanas, abraço não separou nada.** Pouco e
muito abraçado tiveram a mesma probabilidade prevista de infecção.

A diferença só nasce quando o atrito entra. Nos níveis mais altos de conflito
registrados, a distância entre pouco e muito abraçado passa de vinte pontos percentuais.

Não é que o abraço proteja. É que ele muda a inclinação.

E há um segundo achado, mais quieto. Parte do amortecimento associado a **se sentir
amparado** também aparece numa coisa que dá para contar: quantos dias terminaram com um
abraço. Quando os dois entram no mesmo modelo, o coeficiente do apoio percebido encolhe
cerca de um quarto e o intervalo dele passa a cruzar o zero.

Isso não quer dizer que sentir-se apoiado não importe. O intervalo continua compatível
com um efeito real, e encolher não é desaparecer. Quer dizer que uma parte daquilo que a
gente chama de apoio tem corpo, hora e endereço.

E vale dizer com todas as letras o que o estudo não mostra: ele não mostra que abraço
evita resfriado. O efeito forte está na infecção, que é o vírus pegando. Sobre a doença
clínica em si, os achados são pequenos.

*Quantas pessoas você diria que te apoiam, e quantas delas te abraçaram esta semana?*

---

### ⚠️ Limitações do Modelo

- **Nada aqui é causal.** O desenho aleatorizou o vírus, não os abraços. Quem é muito
  abraçado difere de quem é pouco abraçado em coisas que as catorze covariáveis não
  capturam.
- **Divergência que não conseguimos resolver.** Todos os coeficientes centrais do artigo
  foram reproduzidos com três casas decimais a partir dos dados depositados. A atenuação
  que esses coeficientes implicam é de cerca de 27%, e o artigo reporta 32%. Testamos as
  variantes plausíveis do modelo, com e sem covariáveis, com e sem o efeito principal do
  abraço, e nenhuma chega a 32%. Reportamos os dois números e não sabemos qual caminho os
  autores seguiram.
- **Abraço e atrito são autorrelatados**, em catorze ligações noturnas. São mais
  concretos que uma escala de percepção, mas continuam sendo relato.
- **Infecção não é resfriado clínico.** O amortecimento forte está na infecção. O título
  do dia fala em resfriado porque é assim que se fala, e o texto corrige isso.
- **A faixa do gráfico 3 para no percentil 90 de atrito**, porque acima disso restam
  poucas pessoas e a curva viraria extrapolação.
- **Efeito principal e interação não são o mesmo número.** A razão de chances de 0,39 que
  circula vem do modelo sem interação. No modelo com interação, o mesmo termo vale outra
  coisa, porque passa a significar o efeito quando o atrito está na média.
- **A amostra é de adultos de 18 a 55 anos de Pittsburgh**, recrutados por anúncio e
  dispostos a passar seis dias em quarentena. Não se estende a crianças, idosos ou
  pessoas com doença crônica.
- **Dois ensaios combinados**, desenhados originalmente para outra pergunta, sem
  pré-registro desta análise.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]
- Dados: doi:10.5061/dryad.g7b40

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*